### Load packages

In [121]:
import pandas as pd
import os
import zipfile
import argparse
import numpy as np
from typing import List
from typing import Dict
import html5lib


### Set file information

In [122]:
# Zipped data info

data_directory = r"..\data\raw_files"
data_zipfile = "vct_env-epht-cyanobacteria-season-summaries"

data_path = os.path.join(data_directory, data_zipfile + ".zip")
print(data_path)

..\data\raw_files\vct_env-epht-cyanobacteria-season-summaries.zip


In [123]:
# Output file info

output_directory = r"..\data\unified_csvs"
output_file = "vct_unified"

output_path = os.path.join(output_directory, output_file + ".csv")
print(output_path)

..\data\unified_csvs\vct_unified.csv


In [124]:
# Map the files so we know which of the tabs in an excel file we want

# Keys: filenames
# Values: 0 = first tab, 1 = second tab

sheet_map = {
    "ENV_EPHT-cyanobacteria-season-summary-2012.xls": 0,
    "ENV_EPHT-cyanobacteria-season-summary-2013.xls": 0,
    "ENV_EPHT-cyanobacteria-season-summary-2014.xls": 0,
    "ENV_EPHT-cyanobacteria-season-summary-2015.xls": 0,
    "ENV_EPHT-cyanobacteria-season-summary-2016.xlsx": 1,
    "ENV_EPHT-cyanobacteria-season-summary-2017.xlsx": 1,
    "ENV_EPHT-cyanobacteria-season-summary-2018.xlsx": 1,
    "ENV_EPHT-cyanobacteria-season-summary-2019.xlsx": 0,
    "ENV_EPHT-cyanobacteria-season-summary-2020.xlsx": 0,
    "env_epht-cyanobacteria-season-summary-2021.xlsx": 0,
    "env_epht-cyanobacteria-season-summary-2022.xlsx": 0
}

### Define functions

In [125]:
# Function to read in zipped VCT files into a dictionary of dataframes
# (Because the columns aren't all the same in all files, so we need to adjust them before unifying)
# Also, the files are variously xlsx, xls, and csv, so we need to handle that on the fly.

def load_vct_files_from_zip(src: str, sheet_map: Dict[str, int]) -> Dict[str, pd.DataFrame]:

    dfs: Dict[str, pd.DataFrame] = {}

    with zipfile.ZipFile(src, "r") as archive:
        for filename in archive.namelist():
            with archive.open(filename) as file:
                try:
                    if filename.lower().endswith((".xls", ".xlsx")):
                        # choose sheet based on the mapping (default to first sheet)
                        sheet = sheet_map.get(filename, 0)
                        dfs[filename] = pd.read_excel(file, sheet_name=sheet)
                    elif filename.lower().endswith(".html"):
                        dfs[filename] = pd.read_html(file)[0]  # assume 1 table per file
                    elif filename.lower().endswith(".csv"):
                        dfs[filename] = pd.read_csv(file)
                    else:
                        print(f"Skipping unsupported file type: {filename}")
                except Exception as e:
                    print(f"Error reading {filename}: {e}")

    return dfs

### Load algae bloom data

In [126]:
# Call the loader to load each of the files in the zip folder

vct_dfs = load_vct_files_from_zip(data_path, sheet_map)

# Look at a file
vct_dfs["ENV_EPHT-cyanobacteria-season-summary-2012.xls"].head()

c:\Users\hefla\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hefla\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,SampleDate,Lake,Region,Station,Assessment method,Status,Potentially Toxic Cyanobacteria (cells/mL),Potentially Toxic Cyanobacteria Present,Other Algae and Non-Toxic Cyanobacteria Present,"Microcystin (ug/L), if tested","Anatoxin (ug/L), if tested",Collector,Site,SiteID,web status,latitude,longitude,ReportLocationName
0,2012-06-22,Champlain,Main Lake - northern,VT DEC sta 33,Tiered Alert,quantitative,18.0,Anabaena,"diatoms, chrysophytes",not tested,not tested,VTDEC,33,VTDEC33,Generally Safe,44.701167,-73.418167,LTM 33
1,2012-06-22,Champlain,Main Lake - northern,VT DEC sta 36,Tiered Alert,quantitative,8.0,"Anabaena, Aphanizomenon","diatoms, chrysophytes",not tested,not tested,VTDEC,36,VTDEC36,Generally Safe,44.756167,-73.355000,LTM 36
2,2012-06-21,Champlain,Main Lake - central,VT DEC sta 25,Tiered Alert,quantitative,34.0,"Anabaena, Woronichinia/Coelosphaerium","diatoms, chrysophytes",not tested,not tested,VTDEC,25,VTDEC25,Generally Safe,44.582000,-73.281167,LTM 25
3,2012-06-25,Champlain,Main Lake - southern,Arnold Bay,Visual,1a,0.0,NaN,NaN,NaN,NaN,LCC volunteer,101,LCC101,Generally Safe,44.149234,-73.367050,Arnold Bay
4,2012-06-26,Champlain,St. Albans/the islands,Grand Isle State Park,Visual,1b,0.0,NaN,NaN,NaN,NaN,LCC volunteer,102,LCC102,Generally Safe,44.685872,-73.290998,Grand Isle State Park


### Clean up column names

In [127]:
# Define function to preprocess columns so it's easier to reconcile them

def preprocess_column_names(vct_dfs: dict) -> dict:
    """
      - Convert to uppercase
      - Remove spaces
      - Remove underscores
      - Remove forward slashes
      - remove anything from UGL onward
      - standardize latitude ad longitude columns
      - other text changes in clean_col section below
    """
    preprocessed_dfs = {}

    for fname, df in vct_dfs.items():
        new_columns = []
        for col in df.columns:
            # basic cleaning
            clean_col = (
                col.upper()
                .replace(" ", "")
                .replace("_", "")
                .replace("(", "")
                .replace(")", "")
                .replace("/", "")
                .replace("PRESENT", "")
                .replace("24HR", "")
                .replace("H2O", "WATER")
                .replace("SAMPLEDATE", "REPORTDATE")
                .replace("POTENTIALLYTOXICCYANOBACTERIA", "CYANOTAXA")
                .replace("BLOOMINTENSITYALL", "BLOOMINTENSITY")
                .replace("SITENAME", "STATION")
                .replace("SITE#2015", "SITE")
                .replace("SITENUMBER", "SITE")
                .replace("SAMPLING", "SAMPLE")
                .replace("LAKE", "WATERBODY")
                .replace("STATUSWEB", "WEBSTATUS")
                .replace("ANOTOXIN", "ANATOXIN")
                .replace("OTHERALGAEANDNON-TOXICCYANOBACTERIA", "OTHERTAXA")
                .replace("⁰F", "")
                .replace("TEMPERATURE", "TEMP")
            )
            # remove anything after 'UGL'
            if "UGL" in clean_col:
                clean_col = clean_col.split("UGL")[0]
            # standardize LAT/LON
            if clean_col in ("LATITIDE", "LAT"):
                clean_col = "LATITUDE"
            elif clean_col in ("LON", "LONG"):
                clean_col = "LONGITUDE"
            
            if clean_col == "STATUS":
                if fname == "ENV_EPHT-cyanobacteria-season-summary-2013.xls":
                    clean_col = "STATUS_DROP"
                else:
                    clean_col = "BLOOMINTENSITY"
                    
            new_columns.append(clean_col)
        
        df_copy = df.copy()
        df_copy.columns = new_columns
        preprocessed_dfs[fname] = df_copy

    return preprocessed_dfs

In [128]:
# Actually do the preprocessing of column names

vct_dfs_preprocessed = preprocess_column_names(vct_dfs)


### Drop unnecessary columns

In [129]:
# Define the columns that should be dropped

cols_to_drop = [
    "CYLINDROSPERMOPSIN", # Only used in 2015-2019
    "PLANKTONSAMPLEMETHOD", # only used in 2016-2018 and 2020-2022
    "POTENTIALLYTOXICCYANOBACTERIACELLSML", # only used in 2012
    "POTENTIALLYTOXICCYANOCELLSML", # only used in 2014-2015
    "CYANOBACTERIADENSITYCELLSML", # only used in 2016-2019
    "CLINDROSPERMOPSIN", # only used in 2020
    "APPROXIMATEOFFSHORELENGTHBLOOM", # only used in 2021-2022
    "APPROXIMATESHORELENGTHOFBLOOM", # only used in 2021-2022
    "ACCESSTOSAMPLESITE", # only used in 2021
    "METHOD", # only used in 2013-2018 and 2020
    "MONITOREXPERIENCE", # only used in 2014-2020
    "REPORTEREXPERIENCE", # only used in 2013
    "OBJECTID", # only used in 2013
    "BLOOMDISAPPEARED", # only used in 2013
    "STATUSMOD", # only used in 2013
    "WINDDIRECTION", # only used in 2013
    "ALGAECOLOR", # only used in 2013
    "BLOOMEXTENT", # only used in 2013
    "DENSITY", # only used in 2013, 2019, and 2022
    "COLLECTOR", # only used in 2012-2013
    "ADDITIONALDETAILS", # only used in 2019
    "ASSESSMENTMETHOD", # only used in 2012-2019
    "DENSITYCELLSPERML", # only used in 2021
    "CYN", # only used in 2021-2022
    "SITEID", # only used in 2012 as an abbreviation of STATION
    "REPORTTYPE", # only used in 2019
    "SAMPLETYPE", # only used in 2019
    "REPORTLOCATIONNAME", # only used in 2012 and nearly identical to STATION
    "STATUS_DROP", # only used in 2013
    "CYANOTAXACELLSML"
]


In [130]:
# Define function to drop the unnecessary columns

def drop_unnecessary_columns(vct_dfs: dict, cols_to_drop: list) -> dict:

    cleaned_dfs = {}

    for fname, df in vct_dfs.items():
        # Only drop columns that exist in this DataFrame
        cols_in_df = [col for col in cols_to_drop if col in df.columns]
        df_copy = df.drop(columns=cols_in_df)
        cleaned_dfs[fname] = df_copy

    return cleaned_dfs

In [131]:
# ACtually drop the columns

vct_dfs_clean = drop_unnecessary_columns(vct_dfs_preprocessed, cols_to_drop)


### QC

In [132]:
# Look at columns

for i in vct_dfs_clean:
    print(i)
    print(vct_dfs_clean[i].columns)

env-epht-cyanobacteria-season-summary-2022.xlsx
Index(['REPORTDATE', 'REPORTTIME', 'WATERBODY', 'REGION', 'MUNICIPALITY',
       'SITE', 'STATION', 'BLOOMINTENSITY', 'REPORTFREQUENCY', 'AFFILIATION',
       'WEBSTATUS', 'DETAILS', 'WATERTEMP', 'WATERSURFACE', 'CYANOTAXA',
       'OTHERTAXA', 'MICROCYSTIN', 'ANATOXIN', 'LATITUDE', 'LONGITUDE'],
      dtype='object')
ENV_EPHT-cyanobacteria-season-summary-2012.xls
Index(['REPORTDATE', 'WATERBODY', 'REGION', 'STATION', 'BLOOMINTENSITY',
       'CYANOTAXA', 'OTHERTAXA', 'MICROCYSTIN', 'ANATOXIN', 'SITE',
       'WEBSTATUS', 'LATITUDE', 'LONGITUDE'],
      dtype='object')
ENV_EPHT-cyanobacteria-season-summary-2013.xls
Index(['WATERBODY', 'MUNICIPALITY', 'REPORTDATE', 'BLOOMINTENSITY', 'DETAILS',
       'WATERTEMP', 'WATERSURFACE', 'CYANOTAXA', 'OTHERTAXA', 'MICROCYSTIN',
       'ANATOXIN', 'STATION', 'SITE', 'WEBSTATUS', 'LATITUDE', 'LONGITUDE',
       'REPORTFREQUENCY', 'REPORTTIME'],
      dtype='object')
ENV_EPHT-cyanobacteria-season-summ

In [133]:
# Diff schemas

{k: v.columns.tolist() for k, v in vct_dfs_clean.items()}


{'env-epht-cyanobacteria-season-summary-2022.xlsx': ['REPORTDATE',
  'REPORTTIME',
  'WATERBODY',
  'REGION',
  'MUNICIPALITY',
  'SITE',
  'STATION',
  'BLOOMINTENSITY',
  'REPORTFREQUENCY',
  'AFFILIATION',
  'WEBSTATUS',
  'DETAILS',
  'WATERTEMP',
  'WATERSURFACE',
  'CYANOTAXA',
  'OTHERTAXA',
  'MICROCYSTIN',
  'ANATOXIN',
  'LATITUDE',
  'LONGITUDE'],
 'ENV_EPHT-cyanobacteria-season-summary-2012.xls': ['REPORTDATE',
  'WATERBODY',
  'REGION',
  'STATION',
  'BLOOMINTENSITY',
  'CYANOTAXA',
  'OTHERTAXA',
  'MICROCYSTIN',
  'ANATOXIN',
  'SITE',
  'WEBSTATUS',
  'LATITUDE',
  'LONGITUDE'],
 'ENV_EPHT-cyanobacteria-season-summary-2013.xls': ['WATERBODY',
  'MUNICIPALITY',
  'REPORTDATE',
  'BLOOMINTENSITY',
  'DETAILS',
  'WATERTEMP',
  'WATERSURFACE',
  'CYANOTAXA',
  'OTHERTAXA',
  'MICROCYSTIN',
  'ANATOXIN',
  'STATION',
  'SITE',
  'WEBSTATUS',
  'LATITUDE',
  'LONGITUDE',
  'REPORTFREQUENCY',
  'REPORTTIME'],
 'ENV_EPHT-cyanobacteria-season-summary-2014.xls': ['REPORTDATE',


In [134]:
# Print out all columns across all dataframes

all_columns = set()
for df in vct_dfs_clean.values():
    all_columns.update(df.columns.tolist())

print(sorted(all_columns))

['AFFILIATION', 'ANATOXIN', 'BLOOMINTENSITY', 'CYANOTAXA', 'DETAILS', 'LATITUDE', 'LONGITUDE', 'MICROCYSTIN', 'MUNICIPALITY', 'OTHERTAXA', 'REGION', 'REPORTDATE', 'REPORTFREQUENCY', 'REPORTTIME', 'SITE', 'STATION', 'WATERBODY', 'WATERSURFACE', 'WATERTEMP', 'WEBSTATUS']


### Reconcile columns across files

In [135]:
def find_matching_files_by_columns(vct_dfs_clean, reference_file):
    """
    Compare all DataFrames in vct_dfs_clean to the reference_file.
    Returns a list of filenames whose columns match the reference (ignoring order).
    Also prints a report for files that don't match.
    """
    ref_cols = set(vct_dfs_clean[reference_file].columns)
    matching_files = []

    for fname, df in vct_dfs_clean.items():
        if fname == reference_file:
            matching_files.append(fname)
            continue

        df_cols = set(df.columns)
        if df_cols == ref_cols:
            matching_files.append(fname)
        else:
            only_in_ref = ref_cols - df_cols
            only_in_df = df_cols - ref_cols
            print(f"\nFile: {fname}")
            if only_in_ref:
                print(f"  Missing columns compared to {reference_file}: {sorted(only_in_ref)}")
            if only_in_df:
                print(f"  Extra columns compared to {reference_file}: {sorted(only_in_df)}")

    print(f"\nFiles that match {reference_file} by column names: {matching_files}")
    return matching_files


In [136]:
# reference_file = "env-epht-cyanobacteria-season-summary-2021.xlsx"
# reference_file = "ENV_EPHT-cyanobacteria-season-summary-2016.xlsx"
reference_file = "ENV_EPHT-cyanobacteria-season-summary-2014.xls"

matching_files = find_matching_files_by_columns(vct_dfs_clean, reference_file)



File: ENV_EPHT-cyanobacteria-season-summary-2012.xls
  Missing columns compared to ENV_EPHT-cyanobacteria-season-summary-2014.xls: ['AFFILIATION', 'DETAILS', 'MUNICIPALITY', 'REPORTFREQUENCY', 'REPORTTIME', 'WATERSURFACE', 'WATERTEMP']

File: ENV_EPHT-cyanobacteria-season-summary-2013.xls
  Missing columns compared to ENV_EPHT-cyanobacteria-season-summary-2014.xls: ['AFFILIATION', 'REGION']

File: ENV_EPHT-cyanobacteria-season-summary-2019.xlsx
  Missing columns compared to ENV_EPHT-cyanobacteria-season-summary-2014.xls: ['DETAILS', 'REPORTFREQUENCY']

Files that match ENV_EPHT-cyanobacteria-season-summary-2014.xls by column names: ['env-epht-cyanobacteria-season-summary-2022.xlsx', 'ENV_EPHT-cyanobacteria-season-summary-2014.xls', 'ENV_EPHT-cyanobacteria-season-summary-2015.xls', 'ENV_EPHT-cyanobacteria-season-summary-2016.xlsx', 'ENV_EPHT-cyanobacteria-season-summary-2017.xlsx', 'ENV_EPHT-cyanobacteria-season-summary-2018.xlsx', 'ENV_EPHT-cyanobacteria-season-summary-2020.xlsx', 'en

In [137]:
# Check for specific column names

# KEEP THESE COLUMNS even though they're not in every file: 
'''
AFFILIATION
BLOOMINTENSITY
REPORTFREQUENCY
MUNICIPALITY
OTHERTAXA
STATION
REGION
'''

column_to_check = "SAMPLETYPE"

files_with_column = [fname for fname, df in vct_dfs_clean.items() if column_to_check in df.columns]

print(f"Files containing '{column_to_check}':\n", files_with_column)


Files containing 'SAMPLETYPE':
 []


In [138]:
# Fill in REGION in 2013 file (it was missing, but we can fill it using the site numeric values)

# create a site-region lookup table using all of the other files in the VCT dataframe

site_region_lookup = {}

# Build lookup table using every file EXCEPT 2013 file
for fname, df in vct_dfs_clean.items():
    if fname == "ENV_EPHT-cyanobacteria-season-summary-2013.xls":
        continue 

    if "SITE" in df.columns and "REGION" in df.columns:
        # Drop rows where SITE or REGION is missing
        df_clean = df.dropna(subset=["SITE", "REGION"])

        # Keep only the first occurrence of each SITE
        df_clean = df_clean.drop_duplicates(subset="SITE")

        df_clean["SITE"] = df_clean["SITE"].astype(int)

        mapping = df_clean.set_index("SITE")["REGION"].to_dict()
        site_region_lookup.update(mapping)

print("Number of unique SITE keys in lookup table:", len(site_region_lookup))

site_region_lookup

Number of unique SITE keys in lookup table: 396


{2: ' Champlain - South Lake',
 3: ' Champlain - Main Lake South',
 4: ' Champlain - South Lake',
 7: ' Champlain - Main Lake South',
 9: ' Champlain - Main Lake South',
 11: ' Champlain - Inland Sea',
 15: ' Champlain - Main Lake South',
 16: ' Champlain - Main Lake Central',
 18: ' Champlain - Main Lake South',
 19: ' Champlain - Main Lake Central',
 21: ' Champlain - Main Lake Central',
 22: ' Champlain - Main Lake Central',
 23: ' Champlain - Inland Sea',
 25: ' Champlain - Malletts Bay',
 26: 'Champlain - Main Lake North',
 27: ' Champlain - Main Lake Central',
 30: ' Champlain - Missisquoi Bay',
 31: 'Champlain - St. Albans Bay',
 33: ' Champlain - Main Lake Central',
 34: ' Champlain - Inland Sea',
 35: ' Champlain - Inland Sea',
 36: ' Champlain - Main Lake North',
 37: ' Champlain - Inland Sea',
 39: ' Champlain - Main Lake South',
 40: ' Champlain - St. Albans Bay',
 42: ' Champlain - Main Lake Central',
 43: ' Champlain - Main Lake Central',
 44: ' Champlain - Main Lake Cent

In [139]:
# Then fill in REGION in the 2013 file with the correct values, based on SITE number

df_2013 = vct_dfs_clean["ENV_EPHT-cyanobacteria-season-summary-2013.xls"].copy()

df_2013["SITE"] = df_2013["SITE"].astype(int)

# Create REGION column by mapping SITE → REGION
df_2013["REGION"] = df_2013["SITE"].map(site_region_lookup)

# Check for any SITE values that didn't match
print("SITE values with missing REGION mapping:", df_2013[df_2013["REGION"].isna()]["SITE"].value_counts())

# replace existing sites
vct_dfs_sites = vct_dfs_clean.copy()
vct_dfs_sites["ENV_EPHT-cyanobacteria-season-summary-2013.xls"] = df_2013

df_2013.head()

SITE values with missing REGION mapping: SITE
215    2
91     1
112    1
214    1
213    1
Name: count, dtype: int64


,WATERBODY,MUNICIPALITY,REPORTDATE,BLOOMINTENSITY,DETAILS,WATERTEMP,WATERSURFACE,CYANOTAXA,OTHERTAXA,MICROCYSTIN,ANATOXIN,STATION,SITE,WEBSTATUS,LATITUDE,LONGITUDE,REPORTFREQUENCY,REPORTTIME,REGION
0,Lake Champlain,LTM16,2013-06-03,NaN,NaN,0.0,NaN,Aphanizomenon,Diatoms,NaN,NaN,LTM16,16,Generally Safe,44.426000,-73.232000,Routine - Biweekly,NaN,Champlain - Main Lake Central
1,Lake Champlain,NaN,2013-06-03,NaN,NaN,0.0,NaN,Aphanizomenon,"Diatoms, chrysopytes",NaN,NaN,LTM 21,21,Generally Safe,44.474830,-73.231600,Routine - Biweekly,NaN,Champlain - Main Lake Central
2,NaN,NaN,2013-06-04,NaN,NaN,NaN,NaN,no cyanobacteria observed,NaN,NaN,NaN,LTM02,2,Generally Safe,43.714833,-73.383000,Routine - Biweekly,NaN,Champlain - South Lake
3,Lake Champlain,LTM04,2013-06-04,NaN,NaN,0.0,NaN,Aphanizomenon,"Diatoms,green algae,chrysophytes",NaN,NaN,LTM04,4,Generally Safe,43.952000,-73.407833,Routine - Biweekly,NaN,Champlain - South Lake
4,Lake Champlain,LTM25,2013-06-06,NaN,NaN,0.0,NaN,"Anabaena, Aphanizomenon, Woronichinia/Coelosph...",Diatoms,NaN,NaN,LTM25,25,Generally Safe,44.582000,-73.281167,Routine - Biweekly,NaN,Champlain - Malletts Bay


In [140]:
# print(vct_dfs_clean["env-epht-cyanobacteria-season-summary-2021.xlsx"].columns)
print(vct_dfs_sites["ENV_EPHT-cyanobacteria-season-summary-2014.xls"].columns)
print(vct_dfs_sites["ENV_EPHT-cyanobacteria-season-summary-2013.xls"].columns)


Index(['REPORTDATE', 'REPORTTIME', 'WATERBODY', 'REGION', 'MUNICIPALITY',
       'STATION', 'SITE', 'AFFILIATION', 'REPORTFREQUENCY', 'BLOOMINTENSITY',
       'WEBSTATUS', 'DETAILS', 'WATERTEMP', 'WATERSURFACE', 'CYANOTAXA',
       'OTHERTAXA', 'MICROCYSTIN', 'ANATOXIN', 'LATITUDE', 'LONGITUDE'],
      dtype='object')
Index(['WATERBODY', 'MUNICIPALITY', 'REPORTDATE', 'BLOOMINTENSITY', 'DETAILS',
       'WATERTEMP', 'WATERSURFACE', 'CYANOTAXA', 'OTHERTAXA', 'MICROCYSTIN',
       'ANATOXIN', 'STATION', 'SITE', 'WEBSTATUS', 'LATITUDE', 'LONGITUDE',
       'REPORTFREQUENCY', 'REPORTTIME', 'REGION'],
      dtype='object')


### Merge into one unified csv

In [141]:
# Function to merge all files into one CSV using standardized columns from section above

def unify_dfs(vct_dfs: dict, reference_file: str) -> pd.DataFrame:
    """
    Combine all dataframes, using the same columns as the reference DataFrame. 
    Fill missing columns with NaN.

    Parameters:
        vct_dfs: dictionary of dataframes
        reference_file: name of the reference dataframe
    """
    ref_cols = vct_dfs[reference_file].columns.tolist()
    unified_dfs = []

    for fname, df in vct_dfs.items():

        # Fill in missing columns w/ NaN
        missing_cols = [col for col in ref_cols if col not in df.columns]
        for col in missing_cols:
            df[col] = pd.NA

        # 🔎 DEBUG: check for duplicate columns
        if df.columns.duplicated().any():
            print(f"\nDuplicate columns in file: {fname}")
            print(df.columns[df.columns.duplicated()])
            print("All columns:", df.columns.tolist())

        # 🔎 DEBUG: check if columns are unique
        if not df.columns.is_unique:
            print(f"\nNon-unique column index in file: {fname}")
            
        unified_dfs.append(df)

    unified_df = pd.concat(unified_dfs, ignore_index=True)
    
    return unified_df

In [142]:
# Actually do the combining

unified_df = unify_dfs(vct_dfs_sites, "ENV_EPHT-cyanobacteria-season-summary-2014.xls")

print(unified_df.shape)
unified_df.head()

(20601, 20)


,REPORTDATE,REPORTTIME,WATERBODY,REGION,MUNICIPALITY,SITE,STATION,BLOOMINTENSITY,REPORTFREQUENCY,AFFILIATION,WEBSTATUS,DETAILS,WATERTEMP,WATERSURFACE,CYANOTAXA,OTHERTAXA,MICROCYSTIN,ANATOXIN,LATITUDE,LONGITUDE
0,2022-08-01,10:45 AM,Lake Champlain,Champlain - South Lake,Benson,2.0,LTM 02,1b - No Cyanobacteria Observed - brown or turb...,Routine - Biweekly,VT DEC,Generally Safe,NaN,76.0,Rolling,NaN,NaN,NaN,NaN,43.714009,-73.383001
1,2022-08-22,10:00 AM,Lake Champlain,Champlain - South Lake,Benson,2.0,LTM 02,1b - No Cyanobacteria Observed - brown or turb...,Routine - Biweekly,VT DEC,Generally Safe,NaN,77.0,Calm,NaN,NaN,NaN,NaN,43.714009,-73.383001
2,2022-10-05,10:30 AM,Lake Champlain,Champlain - South Lake,Benson,2.0,LTM 02,1b - No Cyanobacteria Observed - brown or turb...,Routine - Biweekly,VT DEC,Generally Safe,NaN,57.0,Calm,NaN,NaN,NaN,NaN,43.714009,-73.383001
3,2022-10-21,10:30 AM,Lake Champlain,Champlain - South Lake,Benson,2.0,LTM 02,1b - No Cyanobacteria Observed - brown or turb...,Routine - Biweekly,VT DEC,Generally Safe,NaN,52.0,Rolling,NaN,NaN,NaN,NaN,43.714009,-73.383001
4,2022-06-23,2:35 PM,Lake Champlain,Champlain - Main Lake South,Panton,3.0,Arnold Bay,1a - No Cyanobacteria Observed - clear water,Routine - Weekly,LCC Volunteer,Generally Safe,NaN,NaN,Rolling,NaN,NaN,NaN,NaN,44.149521,-73.367368


In [143]:
# Force all dates to MM/DD/YYYY format

unified_clean_df = unified_df.copy()

unified_clean_df["REPORTDATE"] = (
    pd.to_datetime(unified_clean_df["REPORTDATE"], errors="coerce")
      .dt.strftime("%m/%d/%Y")
)

In [144]:
# Output the unified csv file

output_df = unified_clean_df.copy()
output_df.to_csv(r"..\data\unified_csvs\vct_unified.csv", index=False)